In [1]:
# Lesson 1.13

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
sqlite_index.count()

144

In [3]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index= sqlite_index,
    llm_client=openai_client,
)

In [4]:
answer = assistant.rag("Where can I find the module 1 homework?")
print(answer)

('You can find the Module 1 homework in the course GitHub repo and on the course management platform.\n\nFor the 2026 cohort, use:\n- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)\n- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md)\n\nThe GitHub homework instructions usually have extra context, while the course platform is mainly for submission and deadlines.', 932)


In [5]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

('You can run Ollama locally by:\n\n1. Installing Ollama from https://ollama.com/download  \n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Starting a model locally:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. Testing that the Ollama server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. Using it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection refused error, restart the server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```', 1085)


In [6]:
answer = assistant.rag("How do I run Olama?")
print(answer)

("I don't know.", 969)


In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
messages = [
    {"role": "user", "content": "How do I run Olama?"}
]

# here's a question, and here's a tool you're allowed to use if you want
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

# If the model decided it didn't need the tool, response.output would contain a normal text/message entry, and response.output_text would just be the answer

response.output

[ResponseFunctionToolCall(arguments='{"query":"Olama run install start use Ollama how do I run Ollama"}', call_id='call_orsKBzivKNTMMQJXYbZtMULe', name='search', type='function_call', id='fc_0547259e72183dea006ac5cb3d11ac87d28b55f45ef0c311f5', caller=None, namespace=None, status='completed')]

In [10]:
import json

call = response.output[0]
print(call)
args = json.loads(call.arguments)
print(args)

results = search(**args)
result_json = json.dumps(results, indent=2)

ResponseFunctionToolCall(arguments='{"query":"Olama run install start use Ollama how do I run Ollama"}', call_id='call_orsKBzivKNTMMQJXYbZtMULe', name='search', type='function_call', id='fc_0547259e72183dea006ac5cb3d11ac87d28b55f45ef0c311f5', caller=None, namespace=None, status='completed')
{'query': 'Olama run install start use Ollama how do I run Ollama'}


In [17]:
response.output[0]

ResponseOutputMessage(id='msg_0547259e72183dea006ac5cb3ea7e887d2b5e7de478201c4a4', content=[ResponseOutputText(annotations=[], text='If you mean **Ollama**, the quickest way to run it is:\n\n```bash\nollama run llama3\n```\n\nThat will:\n\n- download the model if needed,\n- start it locally,\n- open a chat-style prompt in your terminal.\n\n### If Ollama is not installed yet\nInstall it from:\n\nhttps://ollama.com/download\n\n- **macOS**: install the `.pkg`\n- **Windows**: install the `.msi`\n- **Linux**:\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### Test that it’s running\nIn another terminal, try:\n\n```bash\ncurl http://localhost:11434\n```\n\n### If you want to use it from Python\nInstall the client:\n\n```bash\npip install ollama\n```\n\nExample:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model="llama3",\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response["message"]["content"])\n```\n\nIf you want, I can also show you 

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

messages

[{'role': 'user', 'content': 'How do I run Olama?'},
 ResponseFunctionToolCall(arguments='{"query":"Olama run install start use Ollama how do I run Ollama"}', call_id='call_orsKBzivKNTMMQJXYbZtMULe', name='search', type='function_call', id='fc_0547259e72183dea006ac5cb3d11ac87d28b55f45ef0c311f5', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_orsKBzivKNTMMQJXYbZtMULe',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a te

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'If you mean **Ollama**, the quickest way to run it is:\n\n```bash\nollama run llama3\n```\n\nThat will:\n\n- download the model if needed,\n- start it locally,\n- open a chat-style prompt in your terminal.\n\n### If Ollama is not installed yet\nInstall it from:\n\nhttps://ollama.com/download\n\n- **macOS**: install the `.pkg`\n- **Windows**: install the `.msi`\n- **Linux**:\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### Test that it’s running\nIn another terminal, try:\n\n```bash\ncurl http://localhost:11434\n```\n\n### If you want to use it from Python\nInstall the client:\n\n```bash\npip install ollama\n```\n\nExample:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model="llama3",\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response["message"]["content"])\n```\n\nIf you want, I can also show you how to run Ollama with a different model or how to start the server manually with `ollama serve`.'

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1308, 251)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0003468


In [16]:
sqlite_index.close()